# Input tensor assembly from a complex stack

This notebook confirms how `PatchDataset._build_input_tensor` slices a stacked complex array
into primary, secondary and interferogram groups, applies the configured representation to
each group, and concatenates them into a single real-valued input tensor with the channel
count predicted by `InputConfig.total_channels`.

Modules exercised:

- `pipelines.dataset_pipeline.datasets.PatchDataset`
- `configuration.dataset_config.InputConfig`
- `configuration.representation.Representation`

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy             as np
import matplotlib.pyplot as plt

import matplotlib
matplotlib.rcParams["figure.dpi"] = 110
matplotlib.rcParams["image.interpolation"] = "nearest"

SEED = 0
np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
except Exception:
    pass

print("repo root on path:", REPO_ROOT)



## Synthetic stacked inputs

The stack holds one primary pass, two secondaries and three interferograms. Each layer is a
distinct complex pattern so that channels can be traced back to their source layer.


In [ ]:
from tools.logger                       import Logger
from configuration.dataset_config       import InputConfig, OutputConfig
from configuration.representation       import Representation
from pipelines.dataset_pipeline.spatial import Patcher
from pipelines.dataset_pipeline.datasets import PatchDataset

logger = Logger(log_dir="/tmp/dataset_nb_logs", name="nb04", level="WARNING")

n_secondaries    = 2
n_interferograms = 3
n_passes         = 1 + n_secondaries + n_interferograms
H, W             = 32, 32

rng   = np.random.default_rng(SEED)
phase = np.linspace(-np.pi, np.pi, n_passes)
stack = np.empty((n_passes, H, W), dtype=np.complex64)
for p in range(n_passes):
    amp        = (1.0 + 0.3 * p) * np.ones((H, W))
    ramp       = np.linspace(0, phase[p], W)[np.newaxis, :] * np.ones((H, 1))
    stack[p]   = (amp * np.exp(1j * ramp)).astype(np.complex64)

n_gaussians   = 1
gt_parameters = rng.random((n_gaussians * 3, H, W)).astype(np.float32)
print("stack:", stack.shape, "  gt:", gt_parameters.shape)



## Configure inputs and predict the channel layout

We request magnitude+angle for the primary, real+imag for secondaries and angle-only for
interferograms, then check the predicted total channel count and group keys.


In [ ]:
input_config = InputConfig(
    use_primary                   = True,
    primary_representation        = Representation.MAG_ANGLE,
    use_secondaries               = True,
    secondaries_representation    = Representation.REAL_IMAG,
    use_interferograms            = True,
    interferograms_representation = Representation.ANGLE_ONLY,
    use_dem                       = False,
)

total = input_config.total_channels(n_secondaries, n_interferograms)
keys  = input_config.channel_group_keys(n_secondaries, n_interferograms)
print("total channels:", total)
for i, k in enumerate(keys):
    print(f"  channel {i:2d}: {k}")



## Assemble and visualise the input tensor

We construct a `PatchDataset` over a single full-image patch and pull item 0. The resulting
tensor should have `total` channels, ordered primary, then secondaries, then interferograms.


In [ ]:
patcher = Patcher.build(spatial_size=(H, W), patch_size=(H, W), stride=H)

dataset = PatchDataset(
    inputs           = stack,
    gt_parameters    = gt_parameters,
    grid             = patcher,
    input_config     = input_config,
    output_config    = OutputConfig(),
    split_name       = "val",
    n_secondaries    = n_secondaries,
    n_interferograms = n_interferograms,
    n_gaussians      = n_gaussians,
)

input_tensor, gt_tensor = dataset[0]
print("input tensor:", input_tensor.shape, input_tensor.dtype)
print("matches total_channels:", input_tensor.shape[0] == total)

n_show = input_tensor.shape[0]
cols   = n_show
fig, axes = plt.subplots(1, cols, figsize=(1.8 * cols, 2.2), squeeze=False)
for c in range(n_show):
    ax = axes[0, c]
    im = ax.imshow(input_tensor[c], cmap="magma")
    ax.set_title(keys[c] + f"\n[{c}]", fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()



## Verify channel ordering against the source layers

The first two channels are the primary magnitude and angle. We confirm channel 0 equals the
magnitude of the primary layer.


In [ ]:
primary_mag_err = float(np.abs(input_tensor[0] - np.abs(stack[0])).max())
print("primary magnitude channel error:", f"{primary_mag_err:.3e}")

offset_secondaries = input_config.primary_channels_per_pass
print("secondaries start at channel:", offset_secondaries)
print("interferograms start at channel:", offset_secondaries + n_secondaries * input_config.secondaries_channels_per_pass)


## Expected visual outcome

The input tensor should contain exactly `total_channels` panels whose titles match the printed
group keys in order: one `pass/mag` and one `pass/phase` from the primary, four `pass/raw_re_im`
from the two secondaries, and three `ifg/phase` from the interferograms. The primary magnitude
channel should match the source layer to machine precision.